In [1]:
import scanpy as sc
import anndata
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#import scvi
import seaborn as sb

## GSE178341

In [2]:
adata_final =sc.read_h5ad("./output_0531/GSE178341/GSE178341_raw.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [3]:
adata_final.obs['sample'].value_counts()

sample
v2    266604
v3    103511
Name: count, dtype: int64

In [4]:
adata_final

AnnData object with n_obs × n_vars = 370115 × 43113
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'species', 'age', 'intervention', 'location'
    var: 'gene_ids', 'feature_types', 'genome'

In [5]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

/tmp/ipykernel_4093974/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [6]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("./output_0531/GSE178341/GSE178341_raw.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="./output_0531/GSE178341/GSE178341_postR.h5ad")


    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    

/usr/bin/bash: /home/lixiangyu/anaconda3/lib/libtinfo.so.6: no version information available (required by /usr/bin/bash)
/usr/bin/bash: /home/lixiangyu/anaconda3/lib/libtinfo.so.6: no version information available (required by /usr/bin/bash)
/usr/bin/bash: /home/lixiangyu/anaconda3/lib/libtinfo.so.6: no version information available (required by /usr/bin/bash)
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1900: Us

Loading required package: SingleCellExperiment
Loading required package: SummarizedExperiment
Loading required package: MatrixGenerics
Loading required package: matrixStats

Attaching package: ‘MatrixGenerics’

The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOr

In [6]:
adata_final =sc.read_h5ad("./output_0531/GSE178341/GSE178341_postR.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [7]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [8]:
# pp--QC后
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)
print("Number of cells after min_genes >= 200:", adata_final.n_obs)
sc.pp.filter_cells(adata_final, min_counts=500)
print("Number of cells after total_counts >= 500:", adata_final.n_obs)
# mitochondrial UMI fraction <= 50%
adata_final = adata_final[adata_final.obs["pct_counts_mt"] <= 50].copy()
print("Number of cells after pct_counts_mt <= 50:", adata_final.n_obs)


Number of cells before gene filter: 370115


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Number of cells after min_genes >= 200: 370114


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Number of cells after total_counts >= 500: 370114
Number of cells after pct_counts_mt <= 50: 370114


In [9]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

AnnData object with n_obs × n_vars = 370114 × 43113
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'species', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_v2_UMAP', 'decontX_v3_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [10]:
adata_final

AnnData object with n_obs × n_vars = 370114 × 43113
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'species', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_v2_UMAP', 'decontX_v3_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [11]:
adata_final.write("./output_0531/GSE178341/GSE178341_postQC-R.h5ad")
adata_final

AnnData object with n_obs × n_vars = 370114 × 43113
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'species', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_v2_UMAP', 'decontX_v3_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [12]:
adata_final.obs['sample'].value_counts()

sample
v2    266603
v3    103511
Name: count, dtype: int64

# Prepare query data

In [13]:
adata_final = sc.read_h5ad("./output_0531/GSE178341/GSE178341_postQC-R.h5ad")
adata_final

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 370114 × 43113
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'species', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_v2_UMAP', 'decontX_v3_UMAP'
    layers: 'raw_decontXcounts', 'uncorrected_counts'

In [14]:
ensembl_id_df = pd.read_csv("/home/lixiangyu/zr/Annotate/gene_names_to_ensembl_ALLFOUND_allfernandez_no6_withallslysz.csv")
gene_to_ensembl = dict(zip(ensembl_id_df['gene_name'], ensembl_id_df['ensembl_id']))
# Map the variable names in AnnData
adata_final.var['original_gene_names'] = adata_final.var_names
adata_final.var_names = [gene_to_ensembl[gene] if gene in gene_to_ensembl else gene for gene in adata_final.var_names]

In [15]:
import numpy as np
from scipy import sparse

X = adata_final.X

if sparse.issparse(X):
    vals = X.data
else:
    vals = X.ravel()

print("min:", vals.min())
print("max:", vals.max())
print("mean:", vals.mean())
print("has decimals:", np.any(vals[:10000] % 1 != 0))

min: 0
max: 59020
mean: 5.981420877216187
has decimals: False


In [16]:
non_ENSG_vars = adata_final.var_names[~adata_final.var_names.str.startswith('ENSG')]

In [17]:
adata_final.var

,gene_ids,feature_types,genome,mt,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,original_gene_names
ENSG00000243485,ENSG00000243485.5_4,Gene Expression,GRCh37_liftover_v28,False,3,0.000008,99.999189,3.0,RP11-34P13.3
ENSG00000237613,ENSG00000237613.2_2,Gene Expression,GRCh37_liftover_v28,False,0,0.000000,100.000000,0.0,FAM138A
ENSG00000186092,ENSG00000186092.6_4,Gene Expression,GRCh37_liftover_v28,False,1,0.000003,99.999730,1.0,OR4F5
ENSG00000238009,ENSG00000238009.6_5,Gene Expression,GRCh37_liftover_v28,False,1002,0.002734,99.729273,1012.0,RP11-34P13.7
ENSG00000239945,ENSG00000239945.1_5,Gene Expression,GRCh37_liftover_v28,False,15,0.000041,99.995947,15.0,RP11-34P13.8
...,...,...,...,...,...,...,...,...,...
ADT-TCRgd,TCR-gammadelta-B1,Gene Expression,GRCh37_liftover_v28,False,0,0.000000,100.000000,0.0,ADT-TCRgd
ADT-TCRab,TCR-alphabeta-IP26,Gene Expression,GRCh37_liftover_v28,False,0,0.000000,100.000000,0.0,ADT-TCRab
ADT-CD1c,CD1c-L161,Gene Expression,GRCh37_liftover_v28,False,0,0.000000,100.000000,0.0,ADT-CD1c
ADT-CD141,CD141-Thrombomodulin-M80,Gene Expression,GRCh37_liftover_v28,False,0,0.000000,100.000000,0.0,ADT-CD141


In [18]:
# remove not mapped genes
# Convert non_ENSG_vars to a set for faster lookup
non_ENSG_vars_set = set(non_ENSG_vars)

# Filter out the variables that are in non_ENSG_vars_set
adata_final = adata_final[:, ~adata_final.var_names.isin(non_ENSG_vars_set)]

In [19]:
adata_final

View of AnnData object with n_obs × n_vars = 370114 × 31807
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'species', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'original_gene_names'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_v2_UMAP', 'decontX_v3_UMAP'
    layers: 'raw_decontXcounts', 'uncorrected_counts'

In [20]:
adata_final.obs['dataset'].value_counts()

dataset
GSE178341    370114
Name: count, dtype: int64

In [21]:
adata_final.obs = adata_final.obs.rename(columns={'location': 'tissue'})

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [22]:
adata_final

AnnData object with n_obs × n_vars = 370114 × 31807
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'species', 'age', 'intervention', 'tissue', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'original_gene_names'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_v2_UMAP', 'decontX_v3_UMAP'
    layers: 'raw_decontXcounts', 'uncorrected_counts'

In [23]:
adata_final.obs['tissue'].value_counts()

tissue
Unknown    370114
Name: count, dtype: int64

In [24]:
#checks
print("Check how many cells have zero counts for all genes...")
cellwise_sum = adata_final.X.sum(axis=1)
num_cells_zero_counts = (cellwise_sum == 0).sum()
    
if num_cells_zero_counts>0:
    print(num_cells_zero_counts, " cells were found with 0 counts across all genes! Removing these cells now...")
    adata_final = adata_final[cellwise_sum > 0, :]

adata_final = adata_final.copy()


Check how many cells have zero counts for all genes...
24  cells were found with 0 counts across all genes! Removing these cells now...


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [25]:
#normalize
#Perform a clustering for scran normalization in clusters
adata_pp = adata_final.copy()
sc.pp.normalize_total(adata_pp, target_sum=1e6)
sc.pp.log1p(adata_pp)
sc.pp.pca(adata_pp, svd_solver="arpack")
sc.pp.neighbors(adata_pp, n_pcs=30)
sc.tl.leiden(adata_pp, key_added='groups', resolution=0.22)


In [26]:
import rpy2.rinterface_lib.callbacks
import logging

from rpy2.robjects import pandas2ri
import anndata2ri

# Ignore R warning messages
#Note: this can be commented out to get more verbose R output
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR)

# Automatically convert rpy2 outputs to pandas dataframes
pandas2ri.activate()
anndata2ri.activate()
%load_ext rpy2.ipython

/tmp/ipykernel_622173/2253401465.py:13: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [27]:
#Preprocess variables for scran normalization
input_groups = adata_pp.obs['groups']
# data_mat = adata_final.X.T.toarray()
data_mat = adata_final.X.T 

In [28]:
print(type(data_mat))
print(sparse.issparse(data_mat))
print(data_mat.getformat() if sparse.issparse(data_mat) else "dense")

<class 'scipy.sparse._csr.csr_matrix'>
True
csr


In [29]:
%%R -i data_mat -i input_groups -o size_factors
library(scran)
library(Matrix)  # 必须加载此包以支持稀疏矩阵

print("转换稀疏格式")
data_mat <- as(as(data_mat, "TsparseMatrix"), "CsparseMatrix")

# 计算size factors（直接操作稀疏矩阵）
size_factors <- calculateSumFactors(data_mat, clusters=input_groups, min.mean=0.1)
print("计算结束")

[1] "转换稀疏格式"
[1] "计算结束"


Loading required package: SingleCellExperiment
Loading required package: SummarizedExperiment
Loading required package: MatrixGenerics
Loading required package: matrixStats

Attaching package: ‘MatrixGenerics’

The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOr

In [30]:
del adata_pp

In [31]:
adata_final.obs['size_factors'] = size_factors

In [32]:
#Normalize adata 
adata_final.X /= adata_final.obs['size_factors'].values[:,None]
sc.pp.log1p(adata_final)

In [33]:
adata_final.write_h5ad("./output_0531/GSE178341/GSE178341_postQC_normalized.h5ad")


# HVG

In [34]:
adata_new = sc.read_h5ad("./output_0531/GSE178341/GSE178341_postQC_normalized.h5ad")
adata_new

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 370090 × 31807
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'species', 'age', 'intervention', 'tissue', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'original_gene_names'
    uns: 'X_name', 'decontX', 'log1p', 'scDblFinder.threshold'
    obsm: 'decontX_v2_UMAP', 'decontX_v3_UMAP'
    layers: 'raw_decontXcounts', 'uncorrected_counts'

In [35]:
adata_new.X.sum()

858315331.0750185

In [36]:
adata_new.obs['cell_type_level1'] = "unknown"

In [37]:
from scarches.models.base._utils import _validate_var_names

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scvi/_settings.py:63: UserWarning: Since v1.0.0, scvi-tools no longer uses a random seed by default. Run `scvi.settings.seed = 0` to reproduce results from previous versions.
  self.seed = seed
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scvi/_settings.py:70: UserWarning: Setting `dl_pin_memory_gpu_training` is deprecated in v1.0 and will be removed in v1.1. Please pass in `pin_memory` to the data loaders instead.
  self.dl_pin_memory_gpu_training = (
 captum (see https://github.com/pytorch/captum).


In [39]:
adata_new.var_names_make_unique()
varnames_path = "/home/lixiangyu/zr/Annotate/ANNOTATE_new/4_big_integration/train_ref_model_noBasophil_0521/output/ref_model_noBasophil/var_names.csv"###no correct
var_names = np.genfromtxt(varnames_path, delimiter=",", dtype=str)
adata_query = _validate_var_names(adata_new, var_names)

Query data is missing expression data of  160  genes which were contained in the reference dataset.
The missing information will be filled with zeroes.


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


Query data contains expression data of  29967  genes that were not contained in the reference dataset. This information will be removed from the query data object for further processing.
AnnData object with n_obs × n_vars = 370090 × 2000
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'species', 'age', 'intervention', 'tissue', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'cell_type_level1'


In [40]:
adata_query

AnnData object with n_obs × n_vars = 370090 × 2000
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'species', 'age', 'intervention', 'tissue', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'cell_type_level1'

In [41]:
adata_query.write("./output_0531/GSE178341/GSE178341_postQC_normalized_hvg.h5ad")

# train

In [ ]:
#########################在py文件################